In [1]:
import numpy as np
import pandas as pd

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

csv_path = r"..\Data\SyntheticData\2026_03_31_01_52_14\std_synthetic_data_2026_03_31_01_52_14.csv"  # change if needed
df = pd.read_csv(csv_path)

# =========================
# 2. Build features / target
# =========================
target = "Problem_SKU"

# one-hot encode Storage_Size
size_dummies = pd.get_dummies(df["Storage_Size"], prefix="Size", drop_first=True)

# binary encode
defect_linked_num = df["Defect_In_Linked_Receive"].astype(int)

numeric_features = [
    "Global_SKU_Defect_Rate_%_std",
    "ABS_Volume_Difference_std",
    "Aisle_Hold_%_std",
    "#_Pick_Events_std",
    "#_Pick_Events_In_Clique_std",
    "#_Picks_std",
    "#_Picks_In_Clique_std",
    "Time_In_Loc_std",
    "Current_Max_Volume_std",
]



X = pd.concat([df[numeric_features], size_dummies, defect_linked_num], axis=1)
y = df[target].astype(int)

# =========================
# 3. Train/test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_train)
# =========================
# 4. Scoring helper
# =========================
def score_clustering(X_mat, labels):
    labels = np.asarray(labels)
    mask = labels != -1
    unique_clusters = np.unique(labels[mask])

    if len(unique_clusters) < 2:
        return {
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan,
            "n_clusters": len(unique_clusters),
            "noise_rate": float((labels == -1).mean())
        }

    X_use = X_mat[mask]
    y_use = labels[mask]

    return {
        "silhouette": silhouette_score(X_use, y_use),
        "calinski_harabasz": calinski_harabasz_score(X_use, y_use),
        "davies_bouldin": davies_bouldin_score(X_use, y_use),
        "n_clusters": len(unique_clusters),
        "noise_rate": float((labels == -1).mean())
    }

# DB Scan

In [ ]:
# =========================
# 5. DBSCAN
# =========================
dbscan_results = []
best_dbscan = None
best_dbscan_score = -np.inf

for eps in np.arange(0.2, 2.1, 0.1):
    for min_samples in [3, 5, 8, 10]:
        labels = DBSCAN(eps=float(eps), min_samples=min_samples).fit_predict(X_train)
        scores = score_clustering(X_train, labels)

        if scores["n_clusters"] >= 2 and not np.isnan(scores["silhouette"]):
            dbscan_results.append({
                "algorithm": "DBSCAN",
                "eps": round(float(eps), 2),
                "min_samples": min_samples,
                **scores
            })
            if scores["silhouette"] > best_dbscan_score:
                best_dbscan_score = scores["silhouette"]
                best_dbscan = (eps, min_samples, labels, scores)

# GMM Clustering

In [ ]:
# =========================
# 6. MLE clustering via GMM
# =========================
gmm_results = []
best_gmm = None
best_gmm_score = -np.inf

for k in range(2, 11):
    gmm = GaussianMixture(n_components=k, random_state=42)
    labels = gmm.fit_predict(X_train)
    scores = score_clustering(X_train, labels)

    gmm_results.append({
        "algorithm": "GMM (MLE)",
        "n_components": k,
        **scores
    })

    if not np.isnan(scores["silhouette"]) and scores["silhouette"] > best_gmm_score:
        best_gmm_score = scores["silhouette"]
        best_gmm = (k, labels, scores)

# Heirarchical Clustering

In [ ]:
# =========================
# 7. Hierarchical clustering
# =========================
hier_results = []
best_hier = None
best_hier_score = -np.inf

for k in range(2, 11):
    for linkage in ["ward", "complete", "average"]:
        if linkage == "ward":
            model = AgglomerativeClustering(n_clusters=k, linkage=linkage)
        else:
            model = AgglomerativeClustering(n_clusters=k, linkage=linkage)

        labels = model.fit_predict(X_train)
        scores = score_clustering(X_train, labels)

        hier_results.append({
            "algorithm": "Hierarchical",
            "n_clusters": k,
            "linkage": linkage,
            **scores
        })

        if not np.isnan(scores["silhouette"]) and scores["silhouette"] > best_hier_score:
            best_hier_score = scores["silhouette"]
            best_hier = (k, linkage, labels, scores)

# KNN Means

In [ ]:
# =========================
# KMeans clustering
# =========================



kmeans_results = []
best_kmeans = None
best_kmeans_score = -np.inf

for k in range(2, 5):
    print(f"Evaluating KMeans k={k}...")
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_train)
    scores = score_clustering(X_train, labels)

    kmeans_results.append({
        "algorithm": "KMeans",
        "n_clusters": k,
        **scores
    })

    if not np.isnan(scores["silhouette"]) and scores["silhouette"] > best_kmeans_score:
        best_kmeans_score = scores["silhouette"]
        best_kmeans = (k, labels, scores)

print(f"Best KMeans: {best_kmeans[0]} clusters, silhouette={best_kmeans_score:.4f}")

# Plots and Graphs

In [ ]:
# =========================
# 8. Save metrics (now includes KMeans)
# =========================
out_dir = "output"
os.makedirs(out_dir, exist_ok=True)

metrics_df = pd.concat([
    pd.DataFrame(dbscan_results),
    pd.DataFrame(gmm_results),
    pd.DataFrame(kmeans_results),  # <-- ADD THIS
    pd.DataFrame(hier_results)
], ignore_index=True)

metrics_csv = os.path.join(out_dir, "clustering_metrics.csv")
metrics_df.to_csv(metrics_csv, index=False)

# =========================
# 9. Plot cluster assignments (now 4 plots)
# =========================
fig, axes = plt.subplots(1, 4, figsize=(24, 5))  # Changed to 4 subplots

plots = {
    "DBSCAN": best_dbscan[2] if best_dbscan else np.full(len(X_train), -1),
    "GMM (MLE)": best_gmm[1] if best_gmm else np.zeros(len(X_train), dtype=int),
    "KMeans": best_kmeans[1] if best_kmeans else np.zeros(len(X_train), dtype=int),  # <-- ADD THIS
    "Hierarchical": best_hier[2] if best_hier else np.zeros(len(X_train), dtype=int),
}

for ax, (name, labels) in zip(axes, plots.items()):
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="viridis", s=18, alpha=0.8)
    ax.set_title(f"{name}\n({plots[name].max()+1 if plots[name].max()>=0 else 0} clusters)")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.tight_layout()
cluster_plot_path = os.path.join(out_dir, "cluster_pca.png")
plt.savefig(cluster_plot_path, dpi=200, bbox_inches="tight")
plt.close()

# =========================
# 10. KMeans silhouette curve (your snippet, fixed)
# =========================
sil_kmeans = [r["silhouette"] for r in kmeans_results]
plt.figure(figsize=(8, 5))
plt.plot(range(2, 11), sil_kmeans, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.title('KMeans: Optimal Clusters')
plt.grid(True, alpha=0.3)
plt.tight_layout()
kmeans_sil_path = os.path.join(out_dir, "kmeans_silhouette.png")
plt.savefig(kmeans_sil_path, dpi=200, bbox_inches="tight")
plt.close()

# =========================
# 11. Bar chart comparison (now 4 algorithms)
# =========================
best_by_algo = []
for algo in metrics_df["algorithm"].unique():
    sub = metrics_df[metrics_df["algorithm"] == algo].copy()
    best_row = sub.loc[sub["silhouette"].idxmax()]
    best_by_algo.append(best_row)

best_df = pd.DataFrame(best_by_algo)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ["silhouette", "calinski_harabasz", "davies_bouldin"]):
    bars = ax.bar(best_df["algorithm"], best_df[metric], alpha=0.8)
    ax.set_title(metric.replace("_", " ").title())
    ax.set_xlabel("Algorithm")
    ax.tick_params(axis="x", rotation=20)
    # Add value labels on bars
    for bar, val in zip(bars, best_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'{val:.3f}', ha='center', va='bottom')

plt.tight_layout()
metrics_plot_path = os.path.join(out_dir, "clustering_metrics.png")
plt.savefig(metrics_plot_path, dpi=200, bbox_inches="tight")
plt.close()

print("Saved files:")
print(f"  - {metrics_csv}")
print(f"  - {cluster_plot_path}")
print(f"  - {kmeans_sil_path}")
print(f"  - {metrics_plot_path}")
print("\nBest models by silhouette score:")
print(best_df[["algorithm", "n_clusters", "silhouette", "noise_rate"]].round(4))